In [20]:
# test to see if i can link opposite bus stops to each other using their geometry and other data
import polars as pl
import polars.selectors as cs
import polars_st as st
import os

In [21]:
db_uri = os.environ["DATABASE_URL"]

In [22]:
query = """
SELECT id,
       source,
       name,
       ST_AsEWKB(geom) as geometry,
       data
FROM static.stop
WHERE source = 'mta_bus'
"""

dtype=pl.Struct([pl.Field("direction", pl.String)])

df_stops: pl.DataFrame = pl.read_database_uri(query, db_uri).with_columns(pl.col("data").str.json_decode(dtype).name.keep()).unnest("data")
df_stops
# .select("data").row(0)

id,source,name,geometry,direction
str,str,str,binary,str
"""505107""","""mta_bus""","""56 Av/Qbcc""","b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x00\x00\x00\xe0\x84pR\xc0\x00\x00\x00\x00\x84`D@""","""w"""
"""100014""","""mta_bus""","""Bedford Pk Blvd/Grand Concours…","b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x00\x00\x00\x80\xd7xR\xc0\x00\x00\x00\x20\xb0oD@""","""n_w"""
"""100017""","""mta_bus""","""Paul Av/W 205 St""","b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x00\x00\x00\x00\xf1xR\xc0\x00\x00\x00\x20<pD@""","""n_e"""
"""100018""","""mta_bus""","""Paul Av/West Mosholu Pkwy Sout…","b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x00\x00\x00\x80\xb5xR\xc0\x00\x00\x00\xa0\xb0pD@""","""e"""
"""100019""","""mta_bus""","""Grand Concourse/E 138 St""","b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x00\x00\x00\xc0|{R\xc0\x00\x00\x00\xa0\x20hD@""","""n_e"""
…,…,…,…,…
"""985007""","""mta_bus""","""S Broadway/Mclean Av""","b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x00\x00\x00\x20\yR\xc0\x00\x00\x00\x80\xd4uD@""","""s"""
"""985008""","""mta_bus""","""S Broadway/Vark St""","b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x00\x00\x00\xc0`yR\xc0\x00\x00\x00\xa0\xe1vD@""","""s"""
"""985009""","""mta_bus""","""S Broadway/Radford St""","b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x00\x00\x00\x00fyR\xc0\x00\x00\x00\xc0iuD@""","""s"""


In [ ]:
query = """
SELECT *
FROM static.route_stop
WHERE source = 'mta_bus'
"""

dtype = pl.Struct([pl.Field("direction", pl.Int64), pl.Field("headsign", pl.String)])

df_route_stops: pl.DataFrame = pl.read_database_uri(query, db_uri).with_columns(pl.col("data").str.json_decode(dtype=dtype)).unnest("data")
df_route_stops

route_id,stop_id,source,stop_sequence,direction,headsign
str,str,str,i16,i64,str
"""BX27""","""101784""","""mta_bus""",0,1,"""Clason Pt"""
"""B57""","""301875""","""mta_bus""",38,0,"""Maspeth"""
"""B62""","""308277""","""mta_bus""",0,0,"""Astoria"""
"""M5""","""404259""","""mta_bus""",21,1,"""31 St 6 Av"""
"""M5""","""404918""","""mta_bus""",5,1,"""31 St 6 Av"""
…,…,…,…,…,…
"""BXM3""","""985007""","""mta_bus""",2,1,"""Midtown 26 St Via 5 Av"""
"""BXM3""","""985008""","""mta_bus""",1,1,"""Midtown 26 St Via 5 Av"""
"""BXM3""","""985009""","""mta_bus""",3,1,"""Midtown 26 St Via 5 Av"""


## Connect opposite-direction bus stops

**Strategy**: For each bus route, pair stops serving direction 0 with stops serving direction 1. Among all candidate pairs (same route, opposite directions), compute the geographic distance and keep the closest match within a threshold. Most cross-street opposite stops are 15–150 m apart; anything beyond ~500 m is unlikely to be a true opposite stop.


In [26]:
# --- Build route-stop-direction table ---
# df_route_stops columns after unnest: route_id, stop_id, source, stop_sequence, direction, headsign
route_stop_dir = df_route_stops.select(["route_id", "stop_id", "direction"])

# Attach geometry to each (route, stop, direction) row
stops_routed = route_stop_dir.join(df_stops.select(pl.exclude("direction")), left_on="stop_id", right_on="id")

# --- Separate by direction ---
dir0 = stops_routed.filter(pl.col("direction") == 0).rename(
    {"stop_id": "stop_id_0", "geometry": "geom_0"}
)
dir1 = stops_routed.filter(pl.col("direction") == 1).rename(
    {"stop_id": "stop_id_1", "geometry": "geom_1"}
)

# --- Candidate pairs: same route, opposite directions ---
# One pair can appear multiple times (one row per shared route).
# Select only geometry columns and deduplicate to avoid redundant distance calculations.
candidates = (
    dir0.join(dir1, on="route_id")
    .select(["stop_id_0", "geom_0", "stop_id_1", "geom_1"])
    .unique().with_columns(cs.binary().st.to_srid(2263).name.keep())
)

print(f"Candidate pairs (deduplicated): {len(candidates):,}")
candidates.head(3)


Candidate pairs (deduplicated): 390,549


stop_id_0,geom_0,stop_id_1,geom_1
str,binary,str,binary
"""100028""","b""\x01\x01\x00\x00\x20\xd7\x08\x00\x00\x87\xab\xbcM\x87\xb9.A\x9c\xe2\x94\x1b\xc2\xa5\x0dA""","""100115""","b""\x01\x01\x00\x00\x20\xd7\x08\x00\x00\xe8\x08\xd3\xf6]\xc5.Aa\x046\xd6\xa0\xf9\x0dA"""
"""201005""","b""\x01\x01\x00\x00\x20\xd7\x08\x00\x00\x14\x02G\xbd_\xd4,A\xceDb:\xca\xda\x00A""","""201113""","b""\x01\x01\x00\x00\x20\xd7\x08\x00\x00\xeb\xe1\x11\xfe\x063-A\xbd=""\x9d\xbdn\x02A"""
"""200022""","b""\x01\x01\x00\x00\x20\xd7\x08\x00\x00\xe7\xe1U\xe0M\xe5,A\x09\xa3+_a\x10\x05A""","""200051""","b""\x01\x01\x00\x00\x20\xd7\x08\x00\x00\x8aY\xa9gf_-A\xe0k\xd4~\x94U\x05A"""


In [ ]:
# --- Compute distance between each candidate pair ---
MAX_DIST = 1500 # feet

candidates_dist = candidates.with_columns(
    dist=pl.col("geom_0").st.distance(pl.col("geom_1"))
).filter(pl.col("dist") <= MAX_DIST)

print(f"Pairs within {MAX_DIST}ft threshold: {len(candidates_dist):,}")

# --- For each stop (either direction), pick the nearest opposite stop ---
# Direction-0 stops → nearest direction-1 match
best_for_dir0 = (
    candidates_dist.sort("dist")
    .group_by("stop_id_0")
    .agg(
        pl.first("stop_id_1").alias("opposite_stop_id"),
        pl.first("dist")
        # pl.first("dist").alias("dist_deg"),
        # (pl.first("dist") * 111_000).alias("dist_m_approx"),  # rough metres
    )
)

# Direction-1 stops → nearest direction-0 match (flip columns, reuse same frame)
best_for_dir1 = (
    candidates_dist.sort("dist")
    .group_by("stop_id_1")
    .agg(
        pl.first("stop_id_0").alias("opposite_stop_id"),
        pl.first("dist")
        # pl.first("dist").alias("dist_deg"),
        # (pl.first("dist") * 111_000).alias("dist_m_approx"),
    )
    .rename({"stop_id_1": "stop_id_0"})
)

# Combine: if a stop appears in both (it's in dir0 on one route and dir1 on another,
# which shouldn't happen for MTA bus), keep whichever match is closer.
opposite_stops = (
    pl.concat([best_for_dir0, best_for_dir1])
    .sort("dist")
    .unique(subset=["stop_id_0"], keep="first")
    .rename({"stop_id_0": "stop_id"})
    .sort("dist")
)

print(f"\nStops with an opposite match: {len(opposite_stops):,}")
opposite_stops.head(10)


Pairs within 1500ft threshold: 27,768

Stops with an opposite match: 13,405


stop_id,opposite_stop_id,dist
str,str,f64
"""981010""","""405638""",3.491926
"""405638""","""981010""",3.491926
"""801134""","""308472""",5.559227
"""308472""","""801134""",5.559227
"""803058""","""402296""",6.922646
"""402296""","""803058""",6.922646
"""505243""","""804178""",6.985915
"""804178""","""505243""",6.985915
"""203081""","""203470""",7.603584


In [ ]:
# --- Sanity check: visualize a sample of pairs as lines on a map ---
# Build LINESTRING geometries connecting each matched pair so we can see them in st.explore().

sample_pairs = (
    candidates_dist
    .join(opposite_stops.rename({"stop_id": "stop_id_0"}), on="stop_id_0", how="inner")
    # Keep only the pair that was actually chosen as the best match
    .filter(pl.col("stop_id_1") == pl.col("opposite_stop_id"))
    .sample(n=min(200, len(candidates_dist)), seed=32)
)

# Create a line geometry from geom_0 → geom_1 using WKT construction, then parse
line_df = sample_pairs.with_columns(
    geometry=st.from_wkt(pl.format(
        "LINESTRING ({} {}, {} {})",
        pl.col("geom_0").st.x(), pl.col("geom_0").st.y(),
        pl.col("geom_1").st.x(), pl.col("geom_1").st.y()
    )).st.set_srid(2263)
).select(["stop_id_0", "stop_id_1", "dist", "geometry"])

line_df.st.explore("geometry")


/Users/jonah/Github/trainstatus/science/.venv/lib/python3.13/site-packages/lonboard/_geoarrow/ops/reproject.py:116: UserWarning: Input being reprojected to EPSG:4326 CRS.
Lonboard is only able to render data in EPSG:4326 projection.
  warnings.warn(


In [ ]:
# --- Quality summary ---
import altair as alt
alt.data_transformers.enable("vegafusion")

n_total_stops = df_stops.filter(pl.col("direction").is_not_null()).height
n_matched = len(opposite_stops)

print(f"Total mta_bus stops:        {df_stops.height:,}")
print(f"Stops with direction data:  {n_total_stops:,}")
print(f"Stops matched to opposite:  {n_matched:,}  ({n_matched/df_stops.height*100:.1f}%)")
print()
print("Distance distribution (matched pairs):")
print(opposite_stops.select("dist").describe())

# Histogram of distances
opposite_stops['dist'].plot.hist()

Total mta_bus stops:        13,567
Stops with direction data:  13,567
Stops matched to opposite:  13,405  (98.8%)

Distance distribution (matched pairs):
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ dist        │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 13405.0     │
│ null_count ┆ 0.0         │
│ mean       ┆ 248.012742  │
│ std        ┆ 221.301959  │
│ min        ┆ 3.491926    │
│ 25%        ┆ 96.477855   │
│ 50%        ┆ 189.287496  │
│ 75%        ┆ 294.649139  │
│ max        ┆ 1496.510535 │
└────────────┴─────────────┘


alt.Chart(...)

In [ ]:
# see unmatched stops on map
df_stops.join(opposite_stops, how="anti", left_on="id", right_on="stop_id").st.explore()

In [45]:
# see stops that are far away and might be mismatches
opposite_stops.filter(pl.col("dist") > 1000).join(df_stops, left_on="stop_id", right_on="id").st.explore()

In [ ]:
# TODO: this is an example of an incorrectly chosen opposite stop id. might be able to fix it by just lowering max dist threshold
opposite_stops.filter(stop_id="305088")

stop_id,opposite_stop_id,dist
str,str,f64
"""305088""","""304947""",1491.101478
